# 4.2 Cost and performance of the SafeDrive ML and GenAI systems

**Story:** Both SafeDrive systems are live. Finance asks what they cost and how to spend less;
operations wants one dashboard and the right tools to troubleshoot slow or failing requests.

**Task 4.2:** Optimize and manage ML and AI infrastructure costs and performance.

| Skill | What you do here | Status |
|---|---|---|
| 4.2.1 | Instance families; Inference Recommender job 🔒 | 📝 |
| 4.2.2 | CloudWatch Logs Insights on endpoint logs; X-Ray basics (AgentCore Observability is in 4.1b) | ✅ |
| 4.2.3 | CloudWatch dashboard `safedrive-ops` | ✅ |
| 4.2.4 | Capacity for cost, performance, reliability | 📝 |
| 4.2.5 | Tags, Cost Explorer, Budgets, Service Quotas | ✅ |
| 4.2.6 | Purchasing options | 📝 |
| 4.2.7 | Cost per question for three models | ✅ |
| 4.2.8 | Tokens and steps per agent session | ✅ |
| 4.2.9 | Prompt caching, batch inference 🔒, prompt routing, shorter prompts | ✅ |
| 4.2.10 | Token, embedding and vector storage cost patterns | ✅ |

## Key points for the exam

- **FM cost** = input tokens x input price + output tokens x output price. Output tokens cost more.
  Agents and RAG multiply input tokens (history, tool results, retrieved chunks).
- **Cut FM cost:** smaller or distilled model, shorter prompts, `maxTokens`, **prompt caching**
  (repeated prefix), **batch inference** (about 50% cheaper, not real time), **Intelligent Prompt
  Routing** (cheap model for easy prompts), provisioned throughput for steady high volume.
- **SageMaker cost:** right instance family and size, auto scaling, serverless or scale-to-zero for
  spiky traffic, **managed Spot** for training, **SageMaker Savings Plans** for steady use.
- **Cost tools:** cost allocation **tags** → **Cost Explorer** (analyse) → **AWS Budgets** (alert or
  act) → **Cost Anomaly Detection** (unexpected spikes). **Service Quotas** limit usage, not money.
- **Troubleshooting tools:** CloudWatch metrics and alarms, **Logs Insights** (query logs), **X-Ray**
  (trace one request across services), **AgentCore Observability** (agent sessions, traces, spans).
- **Inference Recommender** load-tests a model on several instance types and reports cost and latency.

| Workload | Cheapest fitting option |
|---|---|
| Spiky or low traffic, cold start OK | Serverless inference |
| Steady real-time traffic | Real-time endpoint + auto scaling + Savings Plan |
| Large payloads, long processing, can queue | Asynchronous inference (scales to zero) |
| Whole dataset, no endpoint needed | Batch transform / Bedrock batch inference |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time, uuid, datetime
import pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: the serverless fraud endpoint, the Bedrock agent and the embedded policy chunks
(each returns at once if it already exists).

In [ ]:
ENDPOINT = C.ensure_fraud_endpoint()
AGENT_ID, ALIAS_ID = C.ensure_agent()
chunks = C.ensure_chunks()
state = U.load_state()
print(ENDPOINT, AGENT_ID, len(chunks), "chunks")

Clients used in this notebook.

In [ ]:
sm, smr = U.client("sagemaker"), U.client("sagemaker-runtime")
cw, logs, xray = U.client("cloudwatch"), U.client("logs"), U.client("xray")
bedrock, brt = U.client("bedrock"), U.client("bedrock-runtime")
agent_rt, lam = U.client("bedrock-agent-runtime"), U.client("lambda")
ce, budgets, sq = U.client("ce"), U.client("budgets"), U.client("service-quotas")
pd.set_option("display.max_colwidth", 110)
print("clients ready")

## 4.2.1 Inference instance families

📝 Pick the family by what the model needs, then the smallest size that meets the latency target.

| Family | Examples | Good for | Notes |
|---|---|---|---|
| General purpose (m) | `ml.m5`, `ml.m6i`, `ml.m7i` | Default choice; tree models, small scikit-learn models | Balanced CPU and memory |
| Compute optimized (c) | `ml.c5`, `ml.c6i`, `ml.c7i` | CPU-bound inference, high request rate (XGBoost, linear) | Best price per vCPU |
| Memory optimized (r) | `ml.r5`, `ml.r6i` | Large models or big lookup tables in memory | More GB per vCPU |
| Burstable (t) | `ml.t2`, `ml.t3` | Notebooks, dev and test | Not for steady production load |
| GPU (g) | `ml.g4dn`, `ml.g5`, `ml.g6` | Deep learning and LLM inference | Cost-effective GPU for inference |
| GPU (p) | `ml.p3`, `ml.p4d`, `ml.p5` | Training large models, very large LLMs | Most powerful, most expensive |
| Inferentia (inf) | `ml.inf1`, `ml.inf2` | High-volume deep learning inference | AWS chip; lowest cost per inference if the model compiles for it |
| Graviton (g suffix) | `ml.c6g`, `ml.m6g`, `ml.c7g` | CPU inference on ARM | Up to about 40% better price-performance; needs an ARM container |

- The SafeDrive XGBoost model is small and CPU-bound: `m5`/`c5` or serverless. A GPU would be wasted money.
- Multi-model endpoints, multi-container endpoints and inference components let several models
  share instances.
- **SageMaker Neo** compiles a model for a target hardware; **Inference Recommender** measures which
  instance type is the best fit.

🔒 **Inference Recommender** (a `Default` job takes about 45 minutes and starts several test
endpoints, a few USD). It needs the model name and a sample payload archive in S3.
First find the model behind the fraud endpoint (cheap, always runs).

In [ ]:
cfg_name = sm.describe_endpoint(EndpointName=ENDPOINT)["EndpointConfigName"]
variant = sm.describe_endpoint_config(EndpointConfigName=cfg_name)["ProductionVariants"][0]
FRAUD_MODEL, VARIANT = variant["ModelName"], variant["VariantName"]
print("model:", FRAUD_MODEL, "| variant:", VARIANT)

🔒 Upload a sample payload (`tar.gz` with one CSV request) and start the job.

In [ ]:
if U.expensive("Inference Recommender job"):
    import io, tarfile
    row = U.read_csv(state["test_s3"].split("/", 3)[3], header=None).iloc[0, 1:]
    data, buf = ",".join(str(v) for v in row).encode(), io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as tar:
        info = tarfile.TarInfo("payload.csv"); info.size = len(data)
        tar.addfile(info, io.BytesIO(data))
    U.client("s3").put_object(Bucket=U.bucket(), Key=f"{U.PREFIX}/recommender/payload.tar.gz", Body=buf.getvalue())
    REC_JOB = f"safedrive-fraud-rec-{int(time.time())}"
    sm.create_inference_recommendations_job(
        JobName=REC_JOB, JobType="Default", RoleArn=U.role_arn(), Tags=U.TAGS,
        InputConfig={"ModelName": FRAUD_MODEL, "ContainerConfig": {
            "Domain": "TABULAR", "Task": "CLASSIFICATION", "Framework": "XGBOOST", "FrameworkVersion": "1.7-1",
            "PayloadConfig": {"SamplePayloadUrl": U.s3_uri(f"{U.PREFIX}/recommender/payload.tar.gz"),
                              "SupportedContentTypes": ["text/csv"]},
            "SupportedInstanceTypes": ["ml.m5.large", "ml.c5.large", "ml.c5.xlarge"]}})
    U.save_state(recommender_job=REC_JOB)
    print("started", REC_JOB)

🔒 Read the recommendations when the job is `COMPLETED`: cost per hour, cost per inference, latency
and max invocations for each instance type. Pick the cheapest one that meets your latency target.

In [ ]:
if U.expensive("Inference Recommender results"):
    job = sm.describe_inference_recommendations_job(JobName=U.get_state("recommender_job"))
    print(job["Status"])
    display(pd.DataFrame([{"instance": r["EndpointConfiguration"]["InstanceType"],
                           "cost_per_hour": r["Metrics"]["CostPerHour"],
                           "cost_per_inference": r["Metrics"]["CostPerInference"],
                           "max_invocations_per_min": r["Metrics"]["MaxInvocations"],
                           "model_latency_ms": r["Metrics"]["ModelLatency"]}
                          for r in job.get("InferenceRecommendations", [])]))

## 4.2.2 Troubleshooting tools: Logs Insights and X-Ray

| Tool | Question it answers |
|---|---|
| CloudWatch metrics / alarms | Is something wrong? (errors, latency, CPU, memory, tokens) |
| CloudWatch Logs Insights | What exactly happened? Query the logs of endpoints, jobs, Lambda, Bedrock |
| AWS X-Ray | Where is the time spent? One request traced across services |
| AgentCore Observability | What did the agent do? Sessions, traces and spans (see notebook 4.1b) |
| SageMaker Debugger / profiler | Why is training slow or not converging? |

Endpoint container logs go to the log group `/aws/sagemaker/Endpoints/<endpoint name>`.

Create some log lines: 20 good requests and 2 bad ones (text instead of numbers).

In [ ]:
test = U.read_csv(state["test_s3"].split("/", 3)[3], header=None)
ok, bad, last_error = 0, 0, ""
for i in range(22):
    body = "not,a,valid,claim" if i % 11 == 10 else ",".join(str(v) for v in test.iloc[i, 1:])
    try:
        smr.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv", Body=body); ok += 1
    except ClientError as e:
        bad += 1; last_error = e.response["Error"]["Message"][:150]
print("ok:", ok, "| failed:", bad, "|", last_error)

A helper that runs a Logs Insights query and returns a DataFrame.

In [ ]:
def insights(log_group, query, minutes=60):
    now = int(time.time())
    qid = logs.start_query(logGroupName=log_group, startTime=now - 60 * minutes, endTime=now,
                           queryString=query)["queryId"]
    while True:
        r = logs.get_query_results(queryId=qid)
        if r["status"] not in ("Scheduled", "Running"):
            break
        time.sleep(2)
    return pd.DataFrame([{c["field"]: c["value"] for c in row if c["field"] != "@ptr"} for row in r["results"]])

ENDPOINT_LOGS = f"/aws/sagemaker/Endpoints/{ENDPOINT}"
print(ENDPOINT_LOGS)

Query 1: the newest log lines of the endpoint. Logs arrive after 1-2 minutes; rerun if it is empty.
What a container logs depends on the container (XGBoost logs requests and errors).

In [ ]:
time.sleep(60)
insights(ENDPOINT_LOGS, "fields @timestamp, @logStream, @message | sort @timestamp desc | limit 8")

Query 2: only problems. `filter` with a regular expression, then count per 5 minutes with `stats`.

In [ ]:
insights(ENDPOINT_LOGS, """filter @message like /(?i)error|exception|traceback| 4\\d\\d | 5\\d\\d /
| stats count(*) as problems by bin(5m) as time_window | sort time_window desc""")

Query 3: the tool Lambda of the agent. Lambda writes a `REPORT` line per invocation; Logs Insights
parses it into `@duration`, `@maxMemoryUsed` and `@initDuration` (cold start).

In [ ]:
lam.invoke(FunctionName=C.TOOL_LAMBDA, Payload=json.dumps({"claim_id": "C0000001"}))
time.sleep(30)
insights(f"/aws/lambda/{C.TOOL_LAMBDA}", """filter @type = "REPORT"
| stats count(*) as invocations, avg(@duration) as avg_ms, max(@duration) as max_ms,
  max(@maxMemoryUsed / 1000000) as max_memory_mb, count(@initDuration) as cold_starts""")

### X-Ray basics

- **Trace** = one request end to end. **Segment** = work done by one service. **Subsegment** = a
  downstream call (for example DynamoDB) inside a segment.
- Lambda: switch on **active tracing**. Other code: X-Ray SDK or OpenTelemetry (ADOT).
- The **trace map** shows the services of a request and where errors or latency occur.

Turn on active tracing for the tool Lambda.

In [ ]:
lam.update_function_configuration(FunctionName=C.TOOL_LAMBDA, TracingConfig={"Mode": "Active"})
lam.get_waiter("function_updated_v2").wait(FunctionName=C.TOOL_LAMBDA)
print(lam.get_function_configuration(FunctionName=C.TOOL_LAMBDA)["TracingConfig"])

Invoke the function a few times, then wait for the traces to be processed.

In [ ]:
for i in range(1, 4):
    lam.invoke(FunctionName=C.TOOL_LAMBDA, Payload=json.dumps({"claim_id": f"C000000{i}"}))
time.sleep(45)
print("3 traced invocations sent")

Find the traces with a **filter expression** on the service name. Look at duration and error flags.
If the list is empty, wait a little and rerun. (With Transaction Search enabled only a sample of
traces is indexed, so the list can stay short.)

In [ ]:
now = datetime.datetime.now(datetime.timezone.utc)
summaries = xray.get_trace_summaries(StartTime=now - datetime.timedelta(minutes=10), EndTime=now,
                                     FilterExpression=f'service("{C.TOOL_LAMBDA}")')["TraceSummaries"]
pd.DataFrame([{"id": t["Id"], "duration_s": t.get("Duration"), "error": t.get("HasError"),
               "fault": t.get("HasFault")} for t in summaries])

Open one trace and list its segments. A cold start shows an extra `Init` part.

In [ ]:
if summaries:
    segments = xray.batch_get_traces(TraceIds=[summaries[0]["Id"]])["Traces"][0]["Segments"]
    docs = [json.loads(s["Document"]) for s in segments]
    display(pd.DataFrame([{"segment": d.get("name"), "origin": d.get("origin"),
                           "seconds": round(d.get("end_time", 0) - d.get("start_time", 0), 3),
                           "subsegments": [x.get("name") for x in d.get("subsegments", [])]} for d in docs]))
else:
    print("no traces yet - rerun the previous cell in a minute")

## 4.2.3 Dashboard for both systems

- A CloudWatch dashboard is a JSON document with **widgets** (metric, log query, alarm, text).
- One dashboard can mix services, Regions and accounts.
- First 3 dashboards (up to 50 metrics each) are free; after that about USD 3 per month each.

Define the widgets. A small function keeps each widget to one line.

In [ ]:
def widget(title, metrics, x, y, stat="Sum", width=12):
    return {"type": "metric", "x": x, "y": y, "width": width, "height": 6,
            "properties": {"title": title, "region": U.REGION, "stat": stat, "period": 300, "metrics": metrics}}

ep = ["EndpointName", ENDPOINT, "VariantName", VARIANT]
widgets = [
    {"type": "text", "x": 0, "y": 0, "width": 24, "height": 1,
     "properties": {"markdown": "# SafeDrive operations: fraud model (left) and claims assistant (right)"}},
    widget("Fraud endpoint: invocations and errors", [["AWS/SageMaker", "Invocations"] + ep,
           [".", "Invocation4XXErrors"] + ep, [".", "Invocation5XXErrors"] + ep], 0, 1),
    widget("Fraud endpoint: model latency p90 (microseconds)", [["AWS/SageMaker", "ModelLatency"] + ep], 0, 7, "p90"),
    widget("Bedrock: invocations and throttles", [["AWS/Bedrock", "Invocations", "ModelId", U.TEXT_MODEL],
           [".", "InvocationThrottles", ".", "."]], 12, 1),
    widget("Bedrock: tokens", [["AWS/Bedrock", "InputTokenCount", "ModelId", U.TEXT_MODEL],
           [".", "OutputTokenCount", ".", "."]], 12, 7),
    widget("Agent tool Lambda: errors and duration", [["AWS/Lambda", "Errors", "FunctionName", C.TOOL_LAMBDA],
           [".", "Duration", ".", ".", {"stat": "Average", "yAxis": "right"}]], 0, 13, width=24),
]
print(len(widgets), "widgets")

Create the dashboard. An empty list of validation messages means the JSON is valid.

In [ ]:
r = cw.put_dashboard(DashboardName="safedrive-ops", DashboardBody=json.dumps({"widgets": widgets}))
print("validation messages:", r["DashboardValidationMessages"])
print(f"https://{U.REGION}.console.aws.amazon.com/cloudwatch/home?region={U.REGION}#dashboards/dashboard/safedrive-ops")

Read the dashboard back and list the widget titles. Open the link above to see the charts.

In [ ]:
body = json.loads(cw.get_dashboard(DashboardName="safedrive-ops")["DashboardBody"])
[w["properties"].get("title", "text") for w in body["widgets"]]

## 4.2.4 Capacity: cost, performance, reliability

📝 (Hands-on scaling is in notebook 3.2a.)

| Goal | Options |
|---|---|
| Follow the load | **Auto scaling** (target tracking on `SageMakerVariantInvocationsPerInstance`, step, scheduled) |
| Pay nothing when idle | **Serverless inference**; **scale to zero** with inference components; asynchronous inference with min capacity 0 |
| No cold starts | Minimum instance count ≥ 1, or **provisioned concurrency** for serverless |
| Survive an AZ failure | **2 or more instances** (SageMaker spreads them across AZs); VPC subnets in several AZs |
| Many small models | Multi-model endpoint, inference components (share instances, pack GPUs) |
| Bedrock throughput | On-demand quotas (TPM/RPM) → **cross-Region inference profiles** → **provisioned throughput** |
| Training capacity | Managed Spot with checkpoints; **SageMaker training plans** to reserve GPUs |

- **Quotas** are capacity limits per account and Region (instances per type, tokens per minute).
  Check them before a launch and request increases in **Service Quotas**.
- Scale **out** fast and scale **in** slowly (cooldowns) to avoid flapping.
- Right-size first, then scale: a too-large instance wastes money at every scale step.

## 4.2.5 Cost tools and quotas

- **Cost allocation tags**: tag resources (we use `project=mla-c02-practice`), then **activate** the
  tag key in Billing. It takes up to 24 hours before tagged costs show up, and only for new usage.
- **Cost Explorer**: analyse past cost and forecast. The API costs **USD 0.01 per request** 💰.
- **AWS Budgets**: alert (or run an action) when actual or forecast cost passes a limit.
- **Cost Anomaly Detection**: ML-based alerts on unusual spend, no threshold needed.
- **Service Quotas**: see and raise usage limits.

Check that resources carry the project tag. The Resource Groups Tagging API lists resources by tag.

In [ ]:
tagging = U.client("resourcegroupstaggingapi")
found = tagging.get_resources(TagFilters=[{"Key": "project", "Values": ["mla-c02-practice"]}],
                              ResourcesPerPage=100)["ResourceTagMappingList"]
services = pd.Series([r["ResourceARN"].split(":")[2] for r in found]).value_counts()
print(len(found), "tagged resources (first page)")
services

Activate `project` as a cost allocation tag. This works only in the management (payer) account and
only after the tag has been seen by Billing, so the cell reports an error instead of failing.

In [ ]:
try:
    ce.update_cost_allocation_tags_status(CostAllocationTagsStatus=[{"TagKey": "project", "Status": "Active"}])
    print("tag 'project' is active for cost allocation (data appears within 24 hours)")
except ClientError as e:
    print("could not activate the tag:", e.response["Error"]["Code"], "-", e.response["Error"]["Message"][:120])

💰 Cost Explorer (USD 0.01 per call): cost of the last 30 days **grouped by service**.
Look for SageMaker and Bedrock in the list.

In [ ]:
today = datetime.date.today()
period = {"Start": str(today - datetime.timedelta(days=30)), "End": str(today)}

def cost_table(resp):
    rows = [{"group": g["Keys"][0], "usd": float(g["Metrics"]["UnblendedCost"]["Amount"])}
            for t in resp["ResultsByTime"] for g in t["Groups"]]
    df = pd.DataFrame(rows, columns=["group", "usd"])
    return df.groupby("group")["usd"].sum().sort_values(ascending=False).round(2).head(10)

by_service = ce.get_cost_and_usage(TimePeriod=period, Granularity="MONTHLY", Metrics=["UnblendedCost"],
                                   GroupBy=[{"Type": "DIMENSION", "Key": "SERVICE"}])
cost_table(by_service)

💰 The same query **filtered by the project tag**. It is empty until the tag has been active for a
day. Grouping by `USAGE_TYPE` shows what inside a service costs money (instance hours, tokens, storage).

In [ ]:
by_usage = ce.get_cost_and_usage(
    TimePeriod=period, Granularity="MONTHLY", Metrics=["UnblendedCost"],
    Filter={"Tags": {"Key": "project", "Values": ["mla-c02-practice"]}},
    GroupBy=[{"Type": "DIMENSION", "Key": "USAGE_TYPE"}])
result = cost_table(by_usage)
print(result if len(result) else "no tagged cost data yet (tag not active or less than 24 hours old)")

**AWS Budgets**: a monthly cost budget of USD 25 for the project. Put your email in `BUDGET_EMAIL`
to get an alert at 80% of the limit (actual) and at 100% (forecast). The first two budgets are free.

In [ ]:
BUDGET_EMAIL = ""          # for example "me@example.com"; leave empty for a budget without alerts
BUDGET_NAME = "safedrive-monthly"

def alert(kind, percent):
    return {"Notification": {"NotificationType": kind, "ComparisonOperator": "GREATER_THAN",
                             "Threshold": percent, "ThresholdType": "PERCENTAGE"},
            "Subscribers": [{"SubscriptionType": "EMAIL", "Address": BUDGET_EMAIL}]}

budget = {"BudgetName": BUDGET_NAME, "BudgetType": "COST", "TimeUnit": "MONTHLY",
          "BudgetLimit": {"Amount": "25", "Unit": "USD"},
          "CostFilters": {"TagKeyValue": ["user:project$mla-c02-practice"]}}
notifications = [alert("ACTUAL", 80), alert("FORECASTED", 100)] if BUDGET_EMAIL else []
print(json.dumps(budget, indent=1), "\nalerts:", len(notifications))

Create the budget. If the tag filter is rejected (tag not active yet), the cell creates the budget
for the whole account instead.

In [ ]:
try:
    budgets.create_budget(AccountId=U.account_id(), Budget=budget, NotificationsWithSubscribers=notifications)
except budgets.exceptions.DuplicateRecordException:
    print("budget already exists")
except ClientError as e:
    print("tag filter rejected:", e.response["Error"]["Message"][:120], "-> creating without the filter")
    budget.pop("CostFilters")
    budgets.create_budget(AccountId=U.account_id(), Budget=budget, NotificationsWithSubscribers=notifications)
b = budgets.describe_budget(AccountId=U.account_id(), BudgetName=BUDGET_NAME)["Budget"]
print(b["BudgetName"], b["BudgetLimit"], "| spent so far:", b.get("CalculatedSpend", {}).get("ActualSpend"))

**Service Quotas**: look up the endpoint instance quotas for the `ml.m5` family.
A value of 0 means you must request an increase before you can use that instance type.

In [ ]:
def quotas(service, text):
    pages = sq.get_paginator("list_service_quotas").paginate(ServiceCode=service)
    rows = [{"quota": q["QuotaName"], "value": q["Value"], "adjustable": q["Adjustable"], "code": q["QuotaCode"]}
            for p in pages for q in p["Quotas"] if text.lower() in q["QuotaName"].lower()]
    return pd.DataFrame(rows)

quotas("sagemaker", "ml.m5.large for endpoint usage")

The Bedrock quotas that matter for cost and capacity are **tokens per minute** and **requests per
minute** per model. Here for Nova Lite.

In [ ]:
quotas("bedrock", "Nova Lite").head(8)

📝 More cost controls:
- **Cost Anomaly Detection**: create a monitor (per service or per tag) and a subscription
  (email / SNS, daily or immediate). API: `ce.create_anomaly_monitor`, `ce.create_anomaly_subscription`.
- **Budget actions** can attach a deny policy or stop instances when a limit is passed.
- **Bedrock application inference profiles**: a tagged profile for one application, so that Bedrock
  token costs can be split by team or project.
- **AWS Compute Optimizer** and **Trusted Advisor** flag idle or oversized resources.
- Inside SageMaker: stop idle notebooks (lifecycle configuration), delete unused endpoints.

## 4.2.6 Purchasing options

📝

| Option | Discount | Commitment | Use for |
|---|---|---|---|
| On-demand | none | none | New or unpredictable workloads, short experiments |
| **SageMaker AI Savings Plans** | up to about 64% | USD per hour for 1 or 3 years | Steady SageMaker use (notebooks, training, endpoints, processing); any instance family, size, Region |
| **Managed Spot training** | up to 90% | none; jobs can be interrupted | Training and tuning jobs that use **checkpoints** (`use_spot_instances`, `max_wait`) |
| Bedrock on-demand | none | none | Pay per token; variable traffic |
| **Bedrock batch inference** | about 50% | none; results in hours | Large offline jobs (summaries, classification, embeddings) |
| **Bedrock provisioned throughput** | lower per-token cost at high steady volume | Model units per hour; no commitment, 1 or 6 months | Guaranteed throughput; **required for custom (fine-tuned) models** |
| SageMaker training plans | fixed upfront price | Reserved GPU capacity for a time window | Guaranteed capacity for large training jobs |

- Spot is **not** available for real-time endpoints. Savings Plans do **not** cover Bedrock tokens.
- Compute Savings Plans cover EC2, Fargate and Lambda, **not** SageMaker.

💰 Ask Cost Explorer for a SageMaker Savings Plan recommendation (USD 0.01). With little usage, as in
this practice account, there is usually no recommendation.

In [ ]:
try:
    rec = ce.get_savings_plans_purchase_recommendation(
        SavingsPlansType="SAGEMAKER_SP", TermInYears="ONE_YEAR", PaymentOption="NO_UPFRONT",
        LookbackPeriodInDays="THIRTY_DAYS").get("SavingsPlansPurchaseRecommendation", {})
    print(rec.get("SavingsPlansPurchaseRecommendationSummary") or "no recommendation (not enough steady usage)")
except ClientError as e:
    print("not available:", e.response["Error"]["Code"])

## 4.2.7 Cost of FM inference in production

- On-demand price is per 1,000 (or 1 million) **input** and **output** tokens, different per model.
- The `usage` field of every Converse response gives the exact token counts.
- Also count: guardrail text units, Knowledge Base retrieval and embedding, tool Lambda, logging.

A small price table. ⚠ **Example prices in USD per 1,000 tokens. Check the Amazon Bedrock pricing
page for your Region and edit the numbers.**

In [ ]:
PRICES = {   # example prices, check the pricing page
    U.SMALL_MODEL: {"input": 0.000035, "output": 0.00014},    # Nova Micro
    U.TEXT_MODEL:  {"input": 0.00006,  "output": 0.00024},    # Nova Lite
    U.LARGE_MODEL: {"input": 0.0008,   "output": 0.0032},     # Nova Pro
}

def cost_usd(model, usage):
    p = PRICES[model]
    return usage["inputTokens"] / 1000 * p["input"] + usage["outputTokens"] / 1000 * p["output"]

pd.DataFrame(PRICES).T

Ask one RAG-style question (question + retrieved policy text) and read the usage and the cost.

In [ ]:
context = "\n\n".join(chunks["text"].head(4))
question = "What is the standard deductible and what extra deductible applies to drivers under 25?"
prompt = f"Answer the question using only this policy text.\n\n{context}\n\nQuestion: {question}"

def ask(model, prompt, max_tokens=200):
    r = brt.converse(modelId=model, messages=[{"role": "user", "content": [{"text": prompt}]}],
                     inferenceConfig={"maxTokens": max_tokens, "temperature": 0})
    return r["output"]["message"]["content"][0]["text"], r["usage"], r["metrics"]["latencyMs"]

text, usage, ms = ask(U.TEXT_MODEL, prompt)
print(text, "\n", usage, f"\ncost: ${cost_usd(U.TEXT_MODEL, usage):.6f}")

Compare the three models on the same question: tokens, latency, cost per question, and the monthly
cost for 100,000 questions. Read the answers too: is the cheapest model good enough?

In [ ]:
rows = []
for model in [U.SMALL_MODEL, U.TEXT_MODEL, U.LARGE_MODEL]:
    text, usage, ms = ask(model, prompt)
    c = cost_usd(model, usage)
    rows.append({"model": model, "in": usage["inputTokens"], "out": usage["outputTokens"], "latency_ms": ms,
                 "usd_per_question": round(c, 6), "usd_per_100k_questions": round(c * 100_000, 2),
                 "answer": text[:80]})
pd.DataFrame(rows)

## 4.2.8 Agent resource consumption

- One user question to an agent = **several model calls** (plan, call tool, read result, answer).
  Each call resends the instructions, the tool definitions and the history.
- Measure per session: model calls (steps), input and output tokens, tool calls, time.
- Sources: agent trace (`usage` in each model step), `AWS/Bedrock` token metrics, Lambda metrics of
  the tools, AgentCore Observability for AgentCore agents.

A helper that runs the agent in a session and returns the answer and the usage of each model step.

In [ ]:
def agent_usage(text, session_id):
    start = time.time()
    resp = agent_rt.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID, sessionId=session_id,
                                 inputText=text, enableTrace=True)
    answer, steps, tools = "", [], 0
    for event in resp["completion"]:
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode()
        o = event.get("trace", {}).get("trace", {}).get("orchestrationTrace", {})
        if "modelInvocationOutput" in o:
            steps.append(o["modelInvocationOutput"].get("metadata", {}).get("usage", {}))
        tools += "actionGroupInvocationInput" in o.get("invocationInput", {})
    return {"question": text, "model_calls": len(steps), "tool_calls": tools,
            "inputTokens": sum(s.get("inputTokens", 0) for s in steps),
            "outputTokens": sum(s.get("outputTokens", 0) for s in steps),
            "seconds": round(time.time() - start, 1), "answer": answer[:60]}

print("helper ready")

Run a 3-turn session. Watch `inputTokens` grow from turn to turn: the history is sent again each time.

In [ ]:
session = str(uuid.uuid4())
turns = ["What is the status of claim C0000001?", "And claim C0000002?",
         "Which of the two claims has the higher amount?"]
session_df = pd.DataFrame([agent_usage(t, session) for t in turns])
session_df

Session totals and cost (the agent uses the standard text model). Compare with the single model
call in 4.2.7: an agent turn costs several times more input tokens.

In [ ]:
total = {"inputTokens": int(session_df["inputTokens"].sum()), "outputTokens": int(session_df["outputTokens"].sum())}
print("model calls:", session_df["model_calls"].sum(), "| tool calls:", session_df["tool_calls"].sum())
print(total, f"| session cost: ${cost_usd(U.TEXT_MODEL, total):.5f}")
print("input tokens per model call:", round(total["inputTokens"] / session_df["model_calls"].sum()))

The tool side of the agent: invocations, errors and duration of the Lambda function from CloudWatch
(metrics appear after 1-2 minutes).

In [ ]:
def metric_sum(namespace, metric, dims, stat="Sum", minutes=60):
    now = datetime.datetime.now(datetime.timezone.utc)
    pts = cw.get_metric_statistics(Namespace=namespace, MetricName=metric, Statistics=[stat], Period=60 * minutes,
                                   StartTime=now - datetime.timedelta(minutes=minutes), EndTime=now,
                                   Dimensions=[{"Name": k, "Value": v} for k, v in dims.items()])["Datapoints"]
    return round(sum(p[stat] for p in pts) / (len(pts) if stat == "Average" and pts else 1), 1)

fn = {"FunctionName": C.TOOL_LAMBDA}
print({"invocations": metric_sum("AWS/Lambda", "Invocations", fn), "errors": metric_sum("AWS/Lambda", "Errors", fn),
       "avg_duration_ms": metric_sum("AWS/Lambda", "Duration", fn, "Average")})

## 4.2.9 Manage FM cost with usage optimization

| Technique | Saves | Trade-off |
|---|---|---|
| Smaller or **distilled** model | Price per token | May answer worse; test with your evaluation set |
| Shorter prompt, fewer chunks, summarised history | Input tokens | Less context |
| `maxTokens`, "answer in one sentence" | Output tokens | Shorter answers, risk of truncation |
| **Prompt caching** | Repeated input prefix billed at a lower cache-read rate; lower latency | Prefix must be identical and long enough; cache lives minutes |
| **Batch inference** | About 50% | Not real time |
| **Intelligent Prompt Routing** | Easy prompts go to the cheaper model | Works within one model family |
| Response cache in the application | Whole calls | Stale answers |

### Shorter prompts and output limits

The same question with a wordy prompt and with a tight prompt plus a low `maxTokens`.

In [ ]:
wordy = ("You are a very helpful, friendly and extremely knowledgeable insurance assistant. Please think "
         "carefully and then explain, in as much detail as you can, with background and examples: " + question)
tight = "Answer in one short sentence. " + question
rows = []
for label, p, mt in [("wordy", wordy, 400), ("tight", tight, 60)]:
    text, usage, ms = ask(U.TEXT_MODEL, f"{context}\n\n{p}", mt)
    rows.append({"prompt": label, "in": usage["inputTokens"], "out": usage["outputTokens"], "latency_ms": ms,
                 "usd": round(cost_usd(U.TEXT_MODEL, usage), 6)})
pd.DataFrame(rows)

### Prompt caching

- Put a `cachePoint` after the long, **unchanging** part of the prompt (system prompt, documents, tools).
- First call **writes** the cache (`cacheWriteInputTokens`); later calls with the same prefix **read**
  it (`cacheReadInputTokens`) at a lower price and with lower latency.
- The prefix must reach a model-specific minimum (about 1,000 tokens for Nova) and the cache lives
  about 5 minutes, renewed on each hit. A cache hit is likely but not guaranteed.

Build a long system prompt: the full policy manual (repeated to pass the minimum size).

In [ ]:
docs = [U.read_bytes(k).decode() for k in U.list_keys(f"{C.RAW}/policy_docs/") if k.endswith(".md")]
manual = "\n\n".join(docs * 3)
system = [{"text": "You are the SafeDrive claims assistant. Answer only from this manual.\n\n" + manual},
          {"cachePoint": {"type": "default"}}]
print(len(manual), "characters, roughly", len(manual) // 4, "tokens")

Call the model three times with the same system prompt and different questions. Look at the cache
columns: the first call writes, the next calls read. If the model does not support caching, the cell
prints the error.

In [ ]:
rows = []
try:
    for q in ["What is the glass deductible?", "How long do I have to appeal?", "Is theft covered?"]:
        r = brt.converse(modelId=U.TEXT_MODEL, system=system, inferenceConfig={"maxTokens": 60},
                         messages=[{"role": "user", "content": [{"text": q}]}])
        u = r["usage"]
        rows.append({"question": q, "inputTokens": u["inputTokens"], "cacheWrite": u.get("cacheWriteInputTokens", 0),
                     "cacheRead": u.get("cacheReadInputTokens", 0), "latency_ms": r["metrics"]["latencyMs"]})
except ClientError as e:
    print("prompt caching not supported for this model/Region:", e.response["Error"]["Message"][:150])
pd.DataFrame(rows)

### Intelligent Prompt Routing

- A **prompt router** is one endpoint in front of two models of the same family. It predicts which
  model answers well enough and sends easy prompts to the cheaper one.
- Use the router ARN as `modelId`. The response trace tells which model was used.
- Default routers exist only in some Regions; you can also create your own (`create_prompt_router`).

In [ ]:
try:
    routers = bedrock.list_prompt_routers(type="default")["promptRouterSummaries"]
except ClientError as e:
    routers = []; print("prompt routers not available here:", e.response["Error"]["Code"])
pd.DataFrame([{"name": r["promptRouterName"], "models": [m["modelArn"].split("/")[-1] for m in r["models"]],
               "arn": r["promptRouterArn"]} for r in routers])

Send an easy and a hard prompt through a router (the Nova router if there is one) and see which
model answered each.

In [ ]:
router = next((r for r in routers if "nova" in r["promptRouterName"].lower()), routers[0] if routers else None)
prompts = ["What is a deductible? One sentence.",
           "A customer has two glass claims this year and a theft claim reported 40 days late. "
           "Reason step by step about which claims can be paid and what deductible applies."]
if router:
    for p in prompts:
        try:
            r = brt.converse(modelId=router["promptRouterArn"], messages=[{"role": "user", "content": [{"text": p}]}],
                             inferenceConfig={"maxTokens": 200})
            print(r["trace"]["promptRouter"]["invokedModelId"].split("/")[-1], "<-", p[:50])
        except ClientError as e:
            print("router call failed:", e.response["Error"]["Message"][:150]); break
else:
    print("no prompt router in this Region - read the notes above")

### Batch inference

🔒 `create_model_invocation_job` runs many prompts from a JSONL file in S3 at about half the
on-demand price. A job needs at least 100 records and takes from minutes to hours.
Each line is `{"recordId": ..., "modelInput": <the model's InvokeModel request body>}`.
First write the input file (cheap, always runs): 100 adjuster notes to summarise.

In [ ]:
notes = U.read_jsonl(f"{C.RAW}/adjuster_notes.jsonl")[:100]
lines = [json.dumps({"recordId": n["claim_id"], "modelInput": {
    "messages": [{"role": "user", "content": [{"text": "Summarise this claim note in one sentence: " + n["note"]}]}],
    "inferenceConfig": {"maxTokens": 80}}}) for n in notes]
BATCH_KEY = f"{U.PREFIX}/batch/input/notes.jsonl"
U.client("s3").put_object(Bucket=U.bucket(), Key=BATCH_KEY, Body="\n".join(lines).encode())
print(len(lines), "records ->", U.s3_uri(BATCH_KEY))

🔒 Start the batch job and save its ARN.

In [ ]:
if U.expensive("Bedrock batch inference job"):
    job = bedrock.create_model_invocation_job(
        jobName=f"safedrive-notes-batch-{int(time.time())}", roleArn=U.role_arn(), modelId=U.TEXT_MODEL,
        inputDataConfig={"s3InputDataConfig": {"s3InputFormat": "JSONL", "s3Uri": U.s3_uri(BATCH_KEY)}},
        outputDataConfig={"s3OutputDataConfig": {"s3Uri": U.s3_uri(f"{U.PREFIX}/batch/output/")}},
        tags=[{"key": t["Key"], "value": t["Value"]} for t in U.TAGS])
    U.save_state(batch_job_arn=job["jobArn"])
    print(job["jobArn"])

🔒 Check the status (rerun until `Completed`). The output is a `.jsonl.out` file with one result per
record, plus a `manifest.json.out` with token totals.

In [ ]:
if U.expensive("Bedrock batch job status"):
    j = bedrock.get_model_invocation_job(jobIdentifier=U.get_state("batch_job_arn"))
    print(j["status"], j.get("message", ""))
    print(U.list_keys(f"{U.PREFIX}/batch/output/")[:5])

📝 **Smaller and distilled models:** Bedrock **Model Distillation** trains a small "student" model
from the answers of a large "teacher" model for your use case. You get close to teacher quality at
student price and latency. A distilled (custom) model is served with provisioned throughput or
on-demand custom model deployment, so check that the volume justifies it.

## 4.2.10 AI-specific cost patterns

- **Tokens:** watch input vs output tokens per application; alarm on unusual totals.
- **Embeddings:** you pay per input token, once per chunk at ingestion, again on every re-index, and
  for every query. Changing the embedding model means re-embedding everything.
- **Vector storage:** size = vectors x dimensions x 4 bytes (float32) + metadata + index overhead.
  Fewer dimensions or binary vectors shrink it; so do fewer, larger chunks.
- **Vector store pricing models differ:** S3 Vectors (pay for storage and queries, cheapest for large
  or rarely queried data), OpenSearch Serverless (minimum OCUs run all the time), Aurora pgvector
  (database instance or ACUs).

**Embedding cost of the corpus.** Titan Text Embeddings V2 returns the token count of each input.
Measure a few chunks, then estimate the whole corpus. ⚠ Example price, check the pricing page.

In [ ]:
EMBED_PRICE = 0.00002      # example: USD per 1,000 input tokens for Titan Text Embeddings V2
sample = chunks["text"].head(5)
tokens = []
for t in sample:
    r = brt.invoke_model(modelId=U.EMBED_MODEL, body=json.dumps({"inputText": t, "dimensions": 256}))
    tokens.append(json.loads(r["body"].read())["inputTextTokenCount"])
chars_per_token = sample.str.len().sum() / sum(tokens)
corpus_tokens = chunks["text"].str.len().sum() / chars_per_token
print(f"{chars_per_token:.1f} characters per token | corpus: {len(chunks)} chunks, about {corpus_tokens:,.0f} tokens")
print(f"embedding the corpus once: ${corpus_tokens / 1000 * EMBED_PRICE:.6f}")

Scale it up: SafeDrive wants to index 2 million claim documents (10 chunks each, 300 tokens per
chunk) and serve 50,000 queries a day. Ingestion is a one-time cost; queries are tiny.

In [ ]:
n_chunks, tokens_per_chunk = 2_000_000 * 10, 300
ingest = n_chunks * tokens_per_chunk / 1000 * EMBED_PRICE
queries = 50_000 * 30 * 20 / 1000 * EMBED_PRICE            # 20 tokens per query, 30 days
print(f"one-time ingestion: ${ingest:,.0f} | re-index after a chunking change: another ${ingest:,.0f}")
print(f"query embeddings per month: ${queries:,.2f}")

**Vector storage math:** the same 20 million vectors with 1024, 512 and 256 dimensions (float32) and
as binary vectors (1 bit per dimension). Titan V2 supports all of these.

In [ ]:
rows = []
for dims, bytes_per_dim, label in [(1024, 4, "float32"), (512, 4, "float32"), (256, 4, "float32"), (1024, 1 / 8, "binary")]:
    gb = n_chunks * dims * bytes_per_dim / 1e9
    rows.append({"dimensions": dims, "type": label, "vector_data_gb": round(gb, 1),
                 "vs_1024_float32": f"{gb / (n_chunks * 1024 * 4 / 1e9):.0%}"})
pd.DataFrame(rows)

Fewer dimensions cost some retrieval quality. Check it on your own data: embed the chunks with 256
dimensions and see how often the top hit is the same as with 1024 dimensions.

In [ ]:
import numpy as np
from common import rag
small = np.array([U.embed(t, dimensions=256) for t in chunks["text"]])
full = np.array(chunks["embedding"].tolist())
qs = [q["question"] for q in U.read_jsonl(f"{C.RAW}/qa_eval.jsonl")][:8]
same = sum(rag.top_k(U.embed(q), full, 1)[0][0] == rag.top_k(U.embed(q, dimensions=256), small, 1)[0][0] for q in qs)
print(f"top-1 result identical for {same} of {len(qs)} questions; storage is 4x smaller")

**Token usage alarm.** Metric math adds input and output tokens of the text model; the alarm fires
above 200,000 tokens per hour. This catches runaway agents, loops and prompt-stuffing early.

In [ ]:
def token_metric(metric_id, name):
    return {"Id": metric_id, "ReturnData": False, "MetricStat": {"Period": 3600, "Stat": "Sum", "Metric": {
        "Namespace": "AWS/Bedrock", "MetricName": name, "Dimensions": [{"Name": "ModelId", "Value": U.TEXT_MODEL}]}}}

cw.put_metric_alarm(
    AlarmName="safedrive-bedrock-token-usage", EvaluationPeriods=1, Threshold=200000,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching", Tags=U.TAGS,
    Metrics=[token_metric("tin", "InputTokenCount"), token_metric("tout", "OutputTokenCount"),
             {"Id": "total", "Expression": "tin + tout", "Label": "total tokens per hour", "ReturnData": True}])
a = cw.describe_alarms(AlarmNames=["safedrive-bedrock-token-usage"])["MetricAlarms"][0]
print(a["AlarmName"], "|", a["StateValue"])

Who uses the tokens? If model invocation logging is on (notebook 4.1b), Logs Insights can split token
use by caller identity and model.

In [ ]:
log_group = state.get("bedrock_invocation_log_group")
if log_group:
    try:
        display(insights(log_group, """stats sum(input.inputTokenCount) as input_tokens, sum(output.outputTokenCount) as output_tokens,
          count(*) as calls by identity.arn, modelId | sort input_tokens desc | limit 10""", minutes=24 * 60))
    except ClientError as e:
        print("could not query the log group:", e.response["Error"]["Code"])
else:
    print("invocation logging was not set up (notebook 4.1b) - skipping")

## Exam traps

- **Savings Plans** need a 1- or 3-year commitment and cover SageMaker compute, not Bedrock tokens.
  **Spot** is for training, not for real-time endpoints.
- **Provisioned throughput** is about guaranteed capacity (and custom models); for low or spiky
  traffic on-demand is cheaper.
- **Budgets** alert on money; **Service Quotas** limit resources; **Cost Anomaly Detection** needs no
  threshold. Tags must be **activated** as cost allocation tags before Cost Explorer can use them.
- Prompt caching only helps when the **prefix is identical**; put the changing part (the question) last.
- Batch inference and batch transform are cheapest but **not real time**.

## Quiz

**1.** A team runs nightly training jobs of 3 hours that write checkpoints to S3. How can they cut
training cost the most with little effort?
A) SageMaker Savings Plan  B) Managed Spot training with checkpointing  C) Larger instances  D) Serverless inference

<details><summary>Answer</summary>B. Spot training saves up to 90% and checkpoints make interruptions harmless.</details>

**2.** A chatbot sends the same 3,000-token policy manual as system prompt with every request.
Which feature lowers cost and latency without changing answers?
A) Batch inference  B) Prompt caching with a cache point after the manual
C) Provisioned throughput  D) A larger model

<details><summary>Answer</summary>B. The repeated prefix is read from the cache at a lower price.</details>

**3.** Finance wants an email when the monthly cost of resources tagged `project=claims-ai` is
forecast to pass USD 500.
A) CloudWatch alarm on `Invocations`  B) AWS Budgets cost budget filtered by the tag with a forecast alert
C) Service Quotas  D) AWS Config rule

<details><summary>Answer</summary>B. Budgets supports tag filters and forecast-based notifications.</details>

**4.** 5 million support tickets must be classified by an FM once a week. Results are needed by the
next morning. What is the most cost-effective way?
A) Real-time Converse calls in a loop  B) Provisioned throughput for 6 months
C) Bedrock batch inference job with S3 input and output  D) A GPU endpoint with auto scaling

<details><summary>Answer</summary>C. Batch inference is about 50% cheaper and fits offline workloads.</details>

**5.** A small scikit-learn model gets a few requests per hour, with long idle periods. Occasional
cold starts are acceptable. Which hosting option costs least?
A) `ml.g5.xlarge` real-time endpoint  B) Two `ml.m5.large` instances with auto scaling
C) Serverless inference  D) Multi-AZ real-time endpoint with a Savings Plan

<details><summary>Answer</summary>C. Serverless bills only for the compute used per request and scales to zero.</details>

## Save state

In [ ]:
U.save_state(ops_dashboard="safedrive-ops")
print("saved")

## Cleanup

Kept for later notebooks: the serverless fraud endpoint, the agent and its tool Lambda (no idle cost).
Deleted below: the dashboard, the alarm, the budget, and X-Ray tracing on the Lambda.

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Serverless endpoint `safedrive-fraud` | 4.3a | Redeployed by the catch-up cell |
| Knowledge Base and policy chunks | 4.3b | Rebuilt by the catch-up cell |
| Agent and Lambda | Not used again | - |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

Delete the dashboard.

In [ ]:
U.safe(cw.delete_dashboards, DashboardNames=["safedrive-ops"])

Delete the token usage alarm.

In [ ]:
U.safe(cw.delete_alarms, AlarmNames=["safedrive-bedrock-token-usage"])

Delete the budget.

In [ ]:
U.safe(budgets.delete_budget, AccountId=U.account_id(), BudgetName="safedrive-monthly")

Switch X-Ray active tracing off again on the tool Lambda.

In [ ]:
U.safe(lam.update_function_configuration, FunctionName=C.TOOL_LAMBDA, TracingConfig={"Mode": "PassThrough"})